# Type-Constrained OOD Experiment

**Goal**: Test if CAGP still works when OOD samples are type-constrained (harder setting).

Instead of random tail corruption, we corrupt with entities of the **same type** as the original tail.
This tests whether coverage is just detecting type violations or captures deeper structural patterns.

**Run on Google Colab with GPU runtime.**

In [1]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import roc_auc_score
from collections import defaultdict
import json
import os
import urllib.request
import random

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


In [2]:
CONFIG = {
    'epochs': 50,
    'embedding_dim': 100,
    'batch_size': 2048,
    'lr': 0.001,
    'kl_weight': 0.01,
    'seeds': [42, 123, 456],
}

## 1. Data Loading with Type Information

FB15k-237 has type information via relation patterns.
We infer entity types from the relations they participate in.

In [3]:
def download_fb15k237():
    os.makedirs('data', exist_ok=True)
    base_url = "https://raw.githubusercontent.com/DeepGraphLearning/KnowledgeGraphEmbedding/master/data/FB15k-237"
    for split in ['train', 'test', 'valid']:
        path = f'data/{split}.txt'
        if not os.path.exists(path):
            print(f"Downloading {split}...")
            urllib.request.urlretrieve(f"{base_url}/{split}.txt", path)
    print("Download complete!")

download_fb15k237()

def load_triples(path):
    triples = []
    with open(path) as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 3:
                triples.append((parts[0], parts[1], parts[2]))
    return triples

train = load_triples('data/train.txt')
test = load_triples('data/test.txt')

entities = set()
relations = set()
for h, r, t in train + test:
    entities.add(h)
    entities.add(t)
    relations.add(r)

ent2idx = {e: i for i, e in enumerate(entities)}
rel2idx = {r: i for i, r in enumerate(relations)}
idx2ent = {i: e for e, i in ent2idx.items()}

print(f"Entities: {len(entities)}, Relations: {len(relations)}")
print(f"Train: {len(train)}, Test: {len(test)}")

Download complete!
Entities: 14534, Relations: 237
Train: 272115, Test: 20466


## 2. Infer Entity Types from Relation Patterns

We cluster entities by which relations they participate in (as tail).
Entities with similar relation patterns are considered same "type".

In [4]:
def infer_entity_types(triples, rel2idx):
    """
    Infer entity types based on relation patterns.

    For FB15k-237, we use the relation's domain/range as a proxy for type.
    Entities that appear as tails of the same relations are grouped.
    """
    # Build: relation -> set of entities that appear as tail
    relation_tails = defaultdict(set)
    for h, r, t in triples:
        relation_tails[r].add(t)

    # Simple type: use the most common relation as type signature
    entity_relations = defaultdict(set)
    for h, r, t in triples:
        entity_relations[t].add(r)  # Relations where entity appears as tail

    # Cluster by primary relation (most frequent relation as tail)
    entity_type = {}
    for e, rels in entity_relations.items():
        # Use relation that has fewest tails (most specific)
        primary_rel = min(rels, key=lambda r: len(relation_tails[r]))
        entity_type[e] = primary_rel

    # Handle entities only appearing as heads
    for h, r, t in triples:
        if h not in entity_type:
            entity_type[h] = 'HEAD_ONLY'

    # Build type -> entities mapping
    type_to_entities = defaultdict(list)
    for e, typ in entity_type.items():
        type_to_entities[typ].append(e)

    print(f"Inferred {len(type_to_entities)} entity types")
    print(f"Top 5 types by size:")
    for typ, ents in sorted(type_to_entities.items(), key=lambda x: -len(x[1]))[:5]:
        print(f"  {typ[:50]}: {len(ents)} entities")

    return entity_type, type_to_entities

entity_type, type_to_entities = infer_entity_types(train, rel2idx)

Inferred 226 entity types
Top 5 types by size:
  HEAD_ONLY: 1126 entities
  /tv/tv_program/regular_cast./tv/regular_tv_appeara: 378 entities
  /film/film_subject/films: 352 entities
  /sports/sports_position/players./sports/sports_tea: 347 entities
  /media_common/netflix_genre/titles: 339 entities


## 3. Type-Constrained OOD Generation

In [5]:
def generate_type_constrained_ood(test_triples, entity_type, type_to_entities, ent2idx):
    """
    Generate OOD triples by replacing tail with same-type entity.

    This is HARDER than random corruption because type constraints are satisfied.
    """
    ood_triples = []
    skipped = 0

    for h, r, t in test_triples:
        t_type = entity_type.get(t, 'UNKNOWN')
        candidates = type_to_entities.get(t_type, [])

        # Filter out original tail
        candidates = [c for c in candidates if c != t]

        if len(candidates) == 0:
            # Fallback to random if no same-type candidates
            skipped += 1
            t_ood = random.choice(list(ent2idx.keys()))
        else:
            t_ood = random.choice(candidates)

        ood_triples.append((h, r, t_ood))

    print(f"Generated {len(ood_triples)} type-constrained OOD triples")
    print(f"Skipped (no same-type candidates): {skipped} ({skipped/len(test_triples)*100:.1f}%)")

    return ood_triples

# Preview
random.seed(42)
ood_test = generate_type_constrained_ood(test[:100], entity_type, type_to_entities, ent2idx)
print("\nExamples:")
for i in range(3):
    print(f"  ID:  ({test[i][0][:30]}, {test[i][1][:30]}, {test[i][2][:30]})")
    print(f"  OOD: ({ood_test[i][0][:30]}, {ood_test[i][1][:30]}, {ood_test[i][2][:30]})")
    print()

Generated 100 type-constrained OOD triples
Skipped (no same-type candidates): 6 (6.0%)

Examples:
  ID:  (/m/08966, /travel/travel_destination/cli, /m/05lf_)
  OOD: (/m/08966, /travel/travel_destination/cli, /m/028kb)

  ID:  (/m/01hww_, /music/performance_role/regula, /m/01q99h)
  OOD: (/m/01hww_, /music/performance_role/regula, /m/0gcs9)

  ID:  (/m/09v3jyg, /film/film/release_date_s./fil, /m/0f8l9c)
  OOD: (/m/09v3jyg, /film/film/release_date_s./fil, /m/05qhw)



## 4. CAGP Model

In [6]:
class CAGP(nn.Module):
    """Coverage-Augmented GP-KGE."""

    def __init__(self, num_entities, num_relations, dim):
        super().__init__()
        self.num_entities = num_entities
        self.num_relations = num_relations
        self.dim = dim

        # Entity embeddings (mean + log-variance)
        self.entity_mean = nn.Parameter(torch.randn(num_entities, dim) * 0.1)
        self.entity_logvar = nn.Parameter(torch.zeros(num_entities, dim) - 1.0)

        # Relation embeddings
        self.relation_emb = nn.Embedding(num_relations, dim)
        nn.init.xavier_uniform_(self.relation_emb.weight)

        # Coverage matrix
        self.register_buffer('coverage', torch.zeros(num_entities, num_relations))

        # Learnable alpha
        self.alpha_logit = nn.Parameter(torch.tensor(0.0))

    def forward(self, heads, relations, tails):
        if self.training:
            h = self._sample(heads)
            t = self._sample(tails)
        else:
            h = self.entity_mean[heads]
            t = self.entity_mean[tails]
        r = self.relation_emb(relations)
        return (h * r * t).sum(dim=-1)

    def _sample(self, indices):
        mean = self.entity_mean[indices]
        std = torch.exp(0.5 * self.entity_logvar[indices])
        return mean + std * torch.randn_like(std)

    def get_uncertainty(self, heads, relations, tails):
        # GP variance
        h_var = torch.exp(self.entity_logvar[heads]).mean(dim=-1)
        t_var = torch.exp(self.entity_logvar[tails]).mean(dim=-1)
        gp_var = (h_var + t_var) / 2

        # Coverage uncertainty
        h_cov = self.coverage[heads, relations]
        t_cov = self.coverage[tails, relations]
        cov_unc = 2.0 - h_cov - t_cov

        # Normalize and combine
        gp_var_norm = gp_var / (gp_var.mean() + 1e-8) * cov_unc.mean()
        alpha = torch.sigmoid(self.alpha_logit)

        return alpha * gp_var_norm + (1 - alpha) * cov_unc

    def get_gp_only_uncertainty(self, heads, tails):
        h_var = torch.exp(self.entity_logvar[heads]).mean(dim=-1)
        t_var = torch.exp(self.entity_logvar[tails]).mean(dim=-1)
        return (h_var + t_var) / 2

    def get_coverage_only_uncertainty(self, heads, relations, tails):
        h_cov = self.coverage[heads, relations]
        t_cov = self.coverage[tails, relations]
        return 2.0 - h_cov - t_cov

    def precompute_coverage(self, triples, ent2idx, rel2idx):
        for h, r, t in triples:
            self.coverage[ent2idx[h], rel2idx[r]] = 1.0
            self.coverage[ent2idx[t], rel2idx[r]] = 1.0

    def kl_loss(self):
        kl = -0.5 * torch.sum(
            1 + self.entity_logvar - self.entity_mean.pow(2) - self.entity_logvar.exp()
        )
        return kl / self.num_entities

## 5. Training

In [7]:
from torch.utils.data import DataLoader, TensorDataset

def train_cagp(model, triples, ent2idx, rel2idx, epochs, kl_weight=0.01):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])
    criterion = nn.BCEWithLogitsLoss()

    heads = torch.tensor([ent2idx[h] for h, r, t in triples])
    relations = torch.tensor([rel2idx[r] for h, r, t in triples])
    tails = torch.tensor([ent2idx[t] for h, r, t in triples])

    loader = DataLoader(
        TensorDataset(heads, relations, tails),
        batch_size=CONFIG['batch_size'], shuffle=True
    )

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch_h, batch_r, batch_t in loader:
            batch_h = batch_h.to(device)
            batch_r = batch_r.to(device)
            batch_t = batch_t.to(device)

            pos = model(batch_h, batch_r, batch_t)
            neg_t = torch.randint(0, len(ent2idx), batch_t.shape, device=device)
            neg = model(batch_h, batch_r, neg_t)

            loss = criterion(pos, torch.ones_like(pos)) + criterion(neg, torch.zeros_like(neg))
            loss += kl_weight * model.kl_loss()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(loader):.4f}")

    return model

## 6. Evaluation: Random vs Type-Constrained OOD

In [8]:
def evaluate_ood(model, test_triples, ood_triples, ent2idx, rel2idx, method='cagp'):
    """
    Evaluate OOD detection AUROC.

    Args:
        method: 'cagp', 'gp_only', or 'coverage_only'
    """
    model.eval()

    # ID triples
    id_h = torch.tensor([ent2idx.get(h, 0) for h, r, t in test_triples]).to(device)
    id_r = torch.tensor([rel2idx.get(r, 0) for h, r, t in test_triples]).to(device)
    id_t = torch.tensor([ent2idx.get(t, 0) for h, r, t in test_triples]).to(device)

    # OOD triples
    ood_h = torch.tensor([ent2idx.get(h, 0) for h, r, t in ood_triples]).to(device)
    ood_r = torch.tensor([rel2idx.get(r, 0) for h, r, t in ood_triples]).to(device)
    ood_t = torch.tensor([ent2idx.get(t, 0) for h, r, t in ood_triples]).to(device)

    with torch.no_grad():
        if method == 'cagp':
            id_unc = model.get_uncertainty(id_h, id_r, id_t).cpu().numpy()
            ood_unc = model.get_uncertainty(ood_h, ood_r, ood_t).cpu().numpy()
        elif method == 'gp_only':
            id_unc = model.get_gp_only_uncertainty(id_h, id_t).cpu().numpy()
            ood_unc = model.get_gp_only_uncertainty(ood_h, ood_t).cpu().numpy()
        elif method == 'coverage_only':
            id_unc = model.get_coverage_only_uncertainty(id_h, id_r, id_t).cpu().numpy()
            ood_unc = model.get_coverage_only_uncertainty(ood_h, ood_r, ood_t).cpu().numpy()

    # AUROC: higher uncertainty for OOD should give high AUROC
    labels = np.concatenate([np.zeros(len(id_unc)), np.ones(len(ood_unc))])
    scores = np.concatenate([id_unc, ood_unc])

    return roc_auc_score(labels, scores)

## 7. Main Experiment

In [9]:
results = {
    'random_ood': {'cagp': [], 'gp_only': [], 'coverage_only': []},
    'type_constrained_ood': {'cagp': [], 'gp_only': [], 'coverage_only': []},
}

for seed in CONFIG['seeds']:
    print(f"\n{'='*60}")
    print(f"Seed {seed}")
    print('='*60)

    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    # Initialize model
    model = CAGP(len(ent2idx), len(rel2idx), CONFIG['embedding_dim'])
    model.precompute_coverage(train, ent2idx, rel2idx)

    # Train
    print("\nTraining CAGP...")
    model = train_cagp(model, train, ent2idx, rel2idx, CONFIG['epochs'], CONFIG['kl_weight'])

    print(f"\nLearned alpha: {torch.sigmoid(model.alpha_logit).item():.4f}")

    # Generate OOD samples
    random_ood = [(h, r, random.choice(list(ent2idx.keys()))) for h, r, t in test]
    type_ood = generate_type_constrained_ood(test, entity_type, type_to_entities, ent2idx)

    # Evaluate
    print("\n--- Random OOD ---")
    for method in ['cagp', 'gp_only', 'coverage_only']:
        auroc = evaluate_ood(model, test, random_ood, ent2idx, rel2idx, method)
        results['random_ood'][method].append(auroc)
        print(f"  {method}: {auroc:.4f}")

    print("\n--- Type-Constrained OOD ---")
    for method in ['cagp', 'gp_only', 'coverage_only']:
        auroc = evaluate_ood(model, test, type_ood, ent2idx, rel2idx, method)
        results['type_constrained_ood'][method].append(auroc)
        print(f"  {method}: {auroc:.4f}")


Seed 42

Training CAGP...
  Epoch 10/50, Loss: 1.5280
  Epoch 20/50, Loss: 1.4749
  Epoch 30/50, Loss: 1.4360
  Epoch 40/50, Loss: 1.3959
  Epoch 50/50, Loss: 1.3686

Learned alpha: 0.5000
Generated 20466 type-constrained OOD triples
Skipped (no same-type candidates): 906 (4.4%)

--- Random OOD ---
  cagp: 0.9595
  gp_only: 0.7519
  coverage_only: 0.8205

--- Type-Constrained OOD ---
  cagp: 0.8147
  gp_only: 0.6547
  coverage_only: 0.5696

Seed 123

Training CAGP...
  Epoch 10/50, Loss: 1.5280
  Epoch 20/50, Loss: 1.4748
  Epoch 30/50, Loss: 1.4245
  Epoch 40/50, Loss: 1.3870
  Epoch 50/50, Loss: 1.3169

Learned alpha: 0.5000
Generated 20466 type-constrained OOD triples
Skipped (no same-type candidates): 906 (4.4%)

--- Random OOD ---
  cagp: 0.9591
  gp_only: 0.7533
  coverage_only: 0.8198

--- Type-Constrained OOD ---
  cagp: 0.8149
  gp_only: 0.6545
  coverage_only: 0.5694

Seed 456

Training CAGP...
  Epoch 10/50, Loss: 1.5281
  Epoch 20/50, Loss: 1.4732
  Epoch 30/50, Loss: 1.42

## 8. Summary

In [10]:
print("\n" + "="*70)
print("FINAL RESULTS: FB15k-237")
print("="*70)

print("\n--- Random OOD (Standard Setting) ---")
print(f"{'Method':<20} {'AUROC':<15}")
print("-"*35)
for method in ['coverage_only', 'gp_only', 'cagp']:
    mean = np.mean(results['random_ood'][method])
    std = np.std(results['random_ood'][method])
    print(f"{method:<20} {mean:.4f} ± {std:.4f}")

print("\n--- Type-Constrained OOD (Harder Setting) ---")
print(f"{'Method':<20} {'AUROC':<15}")
print("-"*35)
for method in ['coverage_only', 'gp_only', 'cagp']:
    mean = np.mean(results['type_constrained_ood'][method])
    std = np.std(results['type_constrained_ood'][method])
    print(f"{method:<20} {mean:.4f} ± {std:.4f}")

# Key insight
print("\n--- Key Insight ---")
random_cagp = np.mean(results['random_ood']['cagp'])
type_cagp = np.mean(results['type_constrained_ood']['cagp'])
random_cov = np.mean(results['random_ood']['coverage_only'])
type_cov = np.mean(results['type_constrained_ood']['coverage_only'])

print(f"Coverage drop: {random_cov:.4f} → {type_cov:.4f} ({(random_cov-type_cov)/random_cov*100:.1f}% decrease)")
print(f"CAGP drop: {random_cagp:.4f} → {type_cagp:.4f} ({(random_cagp-type_cagp)/random_cagp*100:.1f}% decrease)")

if type_cagp > type_cov:
    print(f"\n✓ CAGP still outperforms coverage-only under type constraints!")
    print(f"  This proves coverage captures more than just type violations.")
else:
    print(f"\n✗ Coverage-only outperforms CAGP under type constraints.")
    print(f"  This suggests GP component may be capturing type information.")


FINAL RESULTS: FB15k-237

--- Random OOD (Standard Setting) ---
Method               AUROC          
-----------------------------------
coverage_only        0.8205 ± 0.0006
gp_only              0.7522 ± 0.0009
cagp                 0.9595 ± 0.0004

--- Type-Constrained OOD (Harder Setting) ---
Method               AUROC          
-----------------------------------
coverage_only        0.5700 ± 0.0007
gp_only              0.6543 ± 0.0004
cagp                 0.8151 ± 0.0005

--- Key Insight ---
Coverage drop: 0.8205 → 0.5700 (30.5% decrease)
CAGP drop: 0.9595 → 0.8151 (15.0% decrease)

✓ CAGP still outperforms coverage-only under type constraints!
  This proves coverage captures more than just type violations.


In [11]:
# Save results
output = {
    'dataset': 'FB15k-237',
    'config': CONFIG,
    'results': {
        'random_ood': {m: {'mean': float(np.mean(v)), 'std': float(np.std(v))}
                      for m, v in results['random_ood'].items()},
        'type_constrained_ood': {m: {'mean': float(np.mean(v)), 'std': float(np.std(v))}
                                 for m, v in results['type_constrained_ood'].items()},
    }
}

with open('type_constrained_results.json', 'w') as f:
    json.dump(output, f, indent=2)

print("\nResults saved to type_constrained_results.json")
print(json.dumps(output, indent=2))


Results saved to type_constrained_results.json
{
  "dataset": "FB15k-237",
  "config": {
    "epochs": 50,
    "embedding_dim": 100,
    "batch_size": 2048,
    "lr": 0.001,
    "kl_weight": 0.01,
    "seeds": [
      42,
      123,
      456
    ]
  },
  "results": {
    "random_ood": {
      "cagp": {
        "mean": 0.959505531284274,
        "std": 0.000354366082337033
      },
      "gp_only": {
        "mean": 0.7521532754553997,
        "std": 0.0008662584733292153
      },
      "coverage_only": {
        "mean": 0.82054353155184,
        "std": 0.0005759720259370241
      }
    },
    "type_constrained_ood": {
      "cagp": {
        "mean": 0.8151412140769696,
        "std": 0.000498110254597376
      },
      "gp_only": {
        "mean": 0.6543309262056235,
        "std": 0.0004031185414960314
      },
      "coverage_only": {
        "mean": 0.5699670919951207,
        "std": 0.0006872271943859886
      }
    }
  }
}
